In [ ]:
# import pandas as pd
# import numpy as np


# def load_dataset(name_dataset):
#     #leemos el dataset de futbol uruguayo
#     df = pd.read_csv(name_dataset)

#     #nos quedamos con las columnas que nos interesan para el clasificador
#     df = df.drop(columns=["full_time", "competition", "home_ident", "away_ident", "home_country", "away_country", "home_code", "away_code", "home_continent", "away_continent", "continent", "level"])

#     #convertimos las columnas a los tipos de datos correctos
#     df["date"] = pd.to_datetime(df["date"], errors="raise")
#     df["gh"] = pd.to_numeric(df["gh"], errors="raise").astype(int)
#     df["ga"] = pd.to_numeric(df["ga"], errors="raise").astype(int)

#     #ordenamos el dataset por fecha y por equipos, y eliminamos duplicados
#     df = (
#         df.drop_duplicates()
#         .sort_values(["date", "home", "away"], kind="stable")
#         .reset_index(drop=True)
#     )

#     #creamos la columna result, que es el resultado del partido, L si gana el local, V si gana el visitante y E si empatan
#     df["result"] = df.apply(
#         lambda row: "L" if row["gh"] > row["ga"] else ("V" if row["gh"] < row["ga"] else "E"), axis=1
#     )

#     return df

# def load_atributes(dataset: pd.DataFrame, date: pd.Timestamp, years_limit: int = 1, matches_limit: int = 5) -> pd.DataFrame:

#     def get_historial(date, team: str) -> pd.DataFrame:

#         #filtramos el dataset por el equipo y la fecha
#         df = dataset[(dataset["home"] == team) | (dataset["away"] == team)]
#         df = df[(date - pd.DateOffset(years=years_limit) < df["date"]) & (df["date"] < date)]

#         #eliminamos los que sean anteriores a un limite 
#         return df

#     def get_wins(dataset: pd.DataFrame, team: str) -> int:
#         #contamos las victorias del equipo
#         return len(dataset[(dataset["home"] == team) & (dataset["result"] == "L")]) + len(dataset[(dataset["away"] == team) & (dataset["result"] == "V")])

#     def comparar(win_reate_team1: float, win_reate_team2: float) -> str:
#         #comparamos las victorias de los equipos
#         if win_reate_team1 > win_reate_team2:
#             return "L"
#         elif win_reate_team1 < win_reate_team2:
#             return "V"
#         else:
#             return "E"

#     def get_condition(date, team: str) -> int:
#         #obtenemos el historial del equipo
#         df = get_historial(date, team)

#         #me quedo con los ultimos partidos del equipo (si los hay)
#         df = df.tail(matches_limit)

#         #TODO : quizas se puede hacer por puntos en vez de cantidad de victorias, pero por ahora lo dejamos asi
#         return (get_wins(df, team) / len(df) if len(df) > 0 else 0)
    

#     #creamos las nuevas columnas con el historial historico de los equipos
#     dataset["historial"] = comparar(dataset["home"].apply(get_historial, args=(dataset["date"], dataset["home"])), dataset["away"].apply(get_historial, args=(dataset["date"], dataset["away"])))

#     #creamos las nuevas columnas con las condiciones recientes de los equipos
#     dataset["condition_match"] = comparar(dataset["home"].apply(get_condition, args=(dataset["date"], dataset["home"])), dataset["away"].apply(get_condition, args=(dataset["date"], dataset["away"])))  
    
#     return dataset

    







In [ ]:
import pandas as pd
import numpy as np


def load_dataset(name_dataset):
    #leemos el dataset de futbol uruguayo
    df = pd.read_csv(name_dataset)

    #nos quedamos con las columnas que nos interesan para el clasificador
    df = df.drop(columns=["full_time", "competition", "home_ident", "away_ident", "home_country", "away_country", "home_code", "away_code", "home_continent", "away_continent", "continent", "level"])

    #convertimos las columnas a los tipos de datos correctos
    df["date"] = pd.to_datetime(df["date"], errors="raise")
    df["gh"] = pd.to_numeric(df["gh"], errors="raise").astype(int)
    df["ga"] = pd.to_numeric(df["ga"], errors="raise").astype(int)

    #ordenamos el dataset por fecha y por equipos, y eliminamos duplicados
    df = (
        df.drop_duplicates()
        .sort_values(["date", "home", "away"], kind="stable")
        .reset_index(drop=True)
    )

    #creamos la columna result, que es el resultado del partido, L si gana el local, V si gana el visitante y E si empatan
    df["result"] = df.apply(
        lambda row: "L" if row["gh"] > row["ga"] else ("V" if row["gh"] < row["ga"] else "E"), axis=1
    )

    return df

def load_attributes(
    dataset: pd.DataFrame,
    years_limit: int = 1,
    matches_limit: int = 5
) -> pd.DataFrame:

    dataset = dataset.copy()

    def get_historial(
        date: pd.Timestamp,
        team: str
    ) -> pd.DataFrame:

        fecha_inicio = date - pd.DateOffset(
            years=years_limit
        )

        historial = dataset[
            (
                (dataset["home"] == team)
                | (dataset["away"] == team)
            )
            & (dataset["date"] >= fecha_inicio)
            & (dataset["date"] < date)
        ]

        return historial.sort_values("date")

    def get_wins(
        historial: pd.DataFrame,
        team: str
    ) -> int:

        victorias_local = (
            (historial["home"] == team)
            & (historial["result"] == "L")
        ).sum()

        victorias_visitante = (
            (historial["away"] == team)
            & (historial["result"] == "V")
        ).sum()

        return int(
            victorias_local + victorias_visitante
        )

    def get_win_rate(
        historial: pd.DataFrame,
        team: str
    ) -> float:

        if len(historial) == 0:
            return 0.0

        return get_wins(historial, team) / len(historial)

    def comparar(
        win_rate_local: float,
        win_rate_visitante: float
    ) -> str:

        if win_rate_local > win_rate_visitante:
            return "L"
        elif win_rate_local < win_rate_visitante:
            return "V"
        return "E"

    def calcular_atributos_fila(
        row: pd.Series
    ) -> pd.Series:

        fecha = row["date"]
        equipo_local = row["home"]
        equipo_visitante = row["away"]

        historial_local = get_historial(
            fecha,
            equipo_local
        )
        historial_visitante = get_historial(
            fecha,
            equipo_visitante
        )

        tasa_local = get_win_rate(
            historial_local,
            equipo_local
        )
        tasa_visitante = get_win_rate(
            historial_visitante,
            equipo_visitante
        )

        ventaja_historica = comparar(
            tasa_local,
            tasa_visitante
        )

        ultimos_local = historial_local.tail(
            matches_limit
        )
        ultimos_visitante = historial_visitante.tail(
            matches_limit
        )

        condicion_local = get_win_rate(
            ultimos_local,
            equipo_local
        )
        condicion_visitante = get_win_rate(
            ultimos_visitante,
            equipo_visitante
        )

        condicion_partido = comparar(
            condicion_local,
            condicion_visitante
        )

        # historial_suficiente = int(
        #     len(ultimos_local) == matches_limit
        #     and len(ultimos_visitante) == matches_limit
        # )

        return pd.Series({
            "historial": ventaja_historica,
            "condition_match": condicion_partido,
            # "historial_suficiente": historial_suficiente
        })

    nuevos_atributos = dataset.apply(
        calcular_atributos_fila,
        axis=1
    )

    return pd.concat(
        [dataset, nuevos_atributos],
        axis=1
    )

In [83]:
df = load_dataset("futbol_uruguayo.csv")

df_procesado = load_attributes(
    df,
    years_limit=1,
    matches_limit=5
)

print(
    df_procesado[
        [
            "date",
            "home",
            "away",
            "historial",
            "condition_match",
            "historial_suficiente",
            "result"
        ]
    ].head(20)
)

         date                        home                              away  \
0  1932-03-05                 Bella Vista                 Defensor Sporting   
1  1932-03-05                  CA Penarol                       River Plate   
2  1932-03-05             Central Espanol        Rampla Juniors Futbol Club   
3  1932-03-05        Montevideo Wanderers                       Racing Club   
4  1932-03-05                    Nacional  Institucion Atletica Sud America   
5  1932-03-12                  CA Penarol  Institucion Atletica Sud America   
6  1932-03-12           Defensor Sporting                       Racing Club   
7  1932-03-12        Montevideo Wanderers                   Central Espanol   
8  1932-03-12                    Nacional                       River Plate   
9  1932-03-12  Rampla Juniors Futbol Club                       Bella Vista   
10 1932-03-19                  CA Penarol                   Central Espanol   
11 1932-03-19           Defensor Sporting        Ram